# Liver: full composition, spatial organisation and prediction

Run all 24 ROIs and eight held-out mice with the published selected settings.
One shared CNN supplies the K=6 partition. Prediction uses five-seed,
ROI-mean GraphSAGE models; interpretation uses separate patch-pooled models
and 50-step integrated gradients.

**Measured full-fit local runtime: 48 min 47 s**, using an RTX 4070 SUPER
for the CNN and CPU for the GNNs. Colab planning estimate: 1–2.5 h with a T4;
installation and first downloads are excluded. A replay using completed
model checkpoints is faster and does not measure full training. Stage times
are saved below.
CPU-only CNN training takes longer. In Colab, select **Runtime → Change
runtime type → T4 GPU**; restart if the setup cell requests it.

Saved paper values are labelled references. Fresh CNN fits can change cluster
numbers and downstream values. [Supplementary controls](mantpy_liver_controls.ipynb)
and the [exact permutation test](mantpy_liver_permutation.ipynb) are separate.

In [ ]:
from pathlib import Path
import hashlib
from urllib.request import urlopen

_SETUP = Path("experiment_setup.py")
_SETUP_SHA256 = "6faab2e74f2559b2cc76cbb5b0323215a58e15ab2fb355050c55822de4214175"
if not _SETUP.exists():
    with urlopen("https://raw.githubusercontent.com/moeghaf/mantpy_reproducibility/mantpy-api-v1.1.0/notebooks/experiments/experiment_setup.py", timeout=90) as response:
        _setup_bytes = response.read()
    if hashlib.sha256(_setup_bytes).hexdigest() != _SETUP_SHA256:
        raise RuntimeError("Downloaded setup checksum does not match this notebook.")
    _setup_temp = _SETUP.with_name(_SETUP.name + ".download")
    _setup_temp.write_bytes(_setup_bytes)
    _setup_temp.replace(_SETUP)
if hashlib.sha256(_SETUP.read_bytes()).hexdigest() != _SETUP_SHA256:
    raise RuntimeError("Use the setup file matching this notebook release.")
from experiment_setup import prepare_experiment
ASSETS = prepare_experiment("liver")


In [ ]:
import os
from pathlib import Path
import time

import mantpy as mt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from liver_analysis import data, descriptive
import liver_helpers as lh

torch.set_num_threads(min(4, os.cpu_count() or 1))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SELECTED = lh.read_selected()
SEEDS = SELECTED["seeds"]                 # five training seeds: 0–4
RESULTS = Path("results/liver")
RESULTS.mkdir(parents=True, exist_ok=True)
TIMINGS = {}
started = time.perf_counter()
print(f"CNN: {DEVICE}; GNN: CPU; {len(SEEDS)} training seeds")

## 1. Load the full liver cohort

The public Mantpy dataset contains six IMC markers, 24 ROIs and eight mice.
Mouse identity defines the evaluation splits.

In [ ]:
stage = time.perf_counter()
liver = mt.datasets.schistosoma_ecm(
    cache_dir=os.environ.get("MANTPY_CACHE"), progressbar=True, verbose=False
)
images, meta = data.cohort(liver)
assert len(images) == 24 and meta.mouse.nunique() == 8
TIMINGS["load_data"] = time.perf_counter() - stage
display(data.mouse_table(meta))

In [ ]:
roi = data.REPRESENTATIVE["Infected_WT"]
plt.figure(figsize=(6, 6))
plt.imshow(np.arcsinh(images[roi][0]), cmap="magma")
plt.title(f"{roi} · fibronectin, arcsinh intensity")
plt.axis("off")
plt.show()

## 2. Normalize and extract patches

Use the same label-free cohort preprocessing as the paper. This evaluation
is conditional on a cohort-defined ECM partition. It does not hold the
entire image-to-cluster pipeline out by mouse; the separate controls do that.

In [ ]:
stage = time.perf_counter()
norm, bounds = data.normalise(images)      # pooled arcsinh / marker 99.5th percentile
nodes = data.patch_features(norm, meta)   # Mantpy: 10-pixel mean + population SD
nodes["signal"] = data.signal_mask(nodes) # cohort-standardized two-cluster background
windows = lh.make_windows(norm, nodes, patch_size=SELECTED["patch_size"])
TIMINGS["preprocess"] = time.perf_counter() - stage
print(f"{len(nodes):,} complete patches; {int(nodes.signal.sum()):,} signal patches")
display(pd.DataFrame({"marker": data.MARKERS, "bound": bounds}))

## 3. Fit one shared CNN and K=6 partition

The selected encoder uses two convolutional blocks, 96 latent features,
200 contrastive epochs, batch size 1,024 and seed 0. Its embeddings are
standardized once before K-means. No search is run.

`fresh` uses the new encoder fit throughout. Choose `paper` explicitly to
evaluate the classifiers on the deposited partition. Both modes still fit
the selected CNN here, so its fresh output remains inspectable.

In [ ]:
PARTITION_MODE = "fresh"                  # "fresh" or explicitly "paper"
display(pd.Series(SELECTED["cnn"], name="selected CNN settings"))

In [ ]:
stage = time.perf_counter()
cnn, raw_embedding, losses, cnn_record = lh.fit_cnn_cached(
    windows, seed=0, config=SELECTED["cnn"], output_dir=RESULTS, device=DEVICE
)
embedding, labels = lh.cluster_embedding(raw_embedding, seed=0, K=6)
TIMINGS["shared_seed0_cnn"] = time.perf_counter() - stage
del cnn, embedding
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
plt.plot(np.arange(1, len(losses) + 1), losses)
plt.xlabel("Epoch")
plt.ylabel("Contrastive loss")
plt.show()

In [ ]:
nodes = lh.use_partition(nodes, labels, mode=PARTITION_MODE)
partition_dir = RESULTS / PARTITION_MODE
partition_dir.mkdir(parents=True, exist_ok=True)
nodes.to_csv(partition_dir / "nodes.csv.gz", index=False)
meta.to_csv(partition_dir / "metadata.csv", index=False)
lh.write_json(partition_dir / "partition.json", {
    "mode": PARTITION_MODE,
    "source": "fresh selected seed-0 CNN" if PARTITION_MODE == "fresh" else "deposited paper partition",
    "nodes_sha256": lh.file_hash(partition_dir / "nodes.csv.gz"),
    "metadata_sha256": lh.file_hash(partition_dir / "metadata.csv"),
    "cnn_identity": cnn_record["identity"],
    "library_versions": lh.environment(),
    "marker_bounds": bounds.tolist(),
})
remodelled = descriptive.infection_cluster(nodes)["cluster"]
print(f"Partition: {PARTITION_MODE}; infection-associated cluster: ECM {remodelled}")
display(nodes.loc[nodes.signal].groupby("cluster").size().rename("patches"))

In [ ]:
lh.show_patch_map(nodes, roi, outline=remodelled)

## 4. Composition and spatial organisation

Compute per-mouse patch fractions and weight mice equally within each group.
Identify the remodelled cluster with the published abundance rule rather
than assigning it a fixed cluster number.

In [ ]:
# Mantpy pools patch counts within each mouse; infection-cluster selection stays local.
stage = time.perf_counter()
fractions = descriptive.mouse_fractions(nodes)
profiles = descriptive.marker_profiles(nodes)
fractions.to_csv(partition_dir / "composition_per_mouse.csv")
profiles.to_csv(partition_dir / "marker_profiles.csv", index=False)
display(fractions)
fractions.groupby("group")[list(range(6))].mean().reindex(data.GROUPS).plot(
    kind="bar", stacked=True, figsize=(8, 4), colormap="tab10"
)
plt.ylabel("Mean mouse fraction")
plt.xlabel("")
plt.legend(title="ECM cluster", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()
TIMINGS["composition"] = time.perf_counter() - stage

In [ ]:
matrix = profiles[[f"z_mean_{marker}" for marker in data.MARKERS]].to_numpy()
fig, ax = plt.subplots(figsize=(7, 4))
image = ax.imshow(matrix, cmap="RdBu_r", vmin=-2, vmax=2)
ax.set_xticks(range(6), data.MARKERS, rotation=45, ha="right")
ax.set_yticks(range(6), [f"ECM {k}" for k in range(6)])
fig.colorbar(image, ax=ax, label="Marker mean, standardized across clusters")
plt.tight_layout()
plt.show()

In [ ]:
# Public co-occurrence returns mouse-pooled counts/expected values; independent paper checks remain local.
stage = time.perf_counter()
adatas, graphs = lh.make_graphs(nodes, meta, graph_type="delaunay")
cooccurrence = descriptive.cooccurrence(
    nodes, graphs=adatas, infection_cluster_id=remodelled
)
for name, table in cooccurrence.items():
    table.to_csv(partition_dir / f"cooccurrence_{name}.csv", index=False)
display(cooccurrence["pairs"].query("involves_infection_cluster"))
delta = descriptive.pair_matrix(cooccurrence["pairs"], "delta_equal_mouse")
plt.imshow(delta, cmap="RdBu_r", vmin=-np.nanmax(abs(delta)), vmax=np.nanmax(abs(delta)))
plt.xticks(range(6), [f"ECM {k}" for k in range(6)], rotation=45)
plt.yticks(range(6), [f"ECM {k}" for k in range(6)])
plt.colorbar(label="Infected KO − WT co-occurrence, equal mouse means")
plt.show()
TIMINGS["graphs_and_cooccurrence"] = time.perf_counter() - stage

In [ ]:
# Public component_summary uses radius40/min5 and pools components within mice.
stage = time.perf_counter()
regions = descriptive.subgraphs(nodes, adatas, cluster=remodelled)
for name, table in regions.items():
    table.to_csv(partition_dir / f"connected_regions_{name}.csv", index=False)
primary_regions = regions["sensitivity"].query("radius_px == 40 and min_nodes == 5")
display(primary_regions)
fig, ax = plt.subplots()
for position, group in enumerate(data.GROUPS):
    sub = primary_regions.loc[primary_regions.group == group]
    values = sub.median_n_nodes.to_numpy(float)
    finite = np.isfinite(values)
    if finite.any():
        ax.scatter(np.full(finite.sum(), position), values[finite],
                   color=plt.get_cmap("tab10")(position), label=group)
    else:
        ax.text(position, 0.03, "No retained\nregions",
                transform=ax.get_xaxis_transform(), ha="center", va="bottom",
                fontsize=8, color="grey")
ax.set_xticks(range(len(data.GROUPS)), data.GROUPS, rotation=35, ha="right")
ax.set_xlim(-0.5, len(data.GROUPS) - 0.5)
ax.set_ylabel(f"Median ECM {remodelled} component size, patches")
fig.tight_layout()
plt.show()
TIMINGS["connected_regions"] = time.perf_counter() - stage

## 5. Predict infection in every held-out mouse

Fit the fixed two-layer GraphSAGE model for all eight mouse folds and
seeds 0–4. Standardization and class weights use only training ROIs.
The weighted loss is averaged within each ROI, then across ROIs.
Average seed probabilities within an ROI and ROI probabilities within a mouse.

In [ ]:
# Each local outer fold calls mt.nn.train_node_classifier with ROI-mean loss.
display(pd.Series(SELECTED["fixed_gnn"], name="fixed GraphSAGE settings"))
stage = time.perf_counter()
prediction = lh.run_outer_folds(
    graphs, meta, seeds=SEEDS, config=SELECTED["fixed_gnn"],
    objective="roi_mean", output_dir=partition_dir / "prediction"
)
lh.save_outer_outputs(prediction, partition_dir / "prediction")
prediction_metrics = lh.metric_table(prediction["ensemble"], "GraphSAGE · ROI-mean")
TIMINGS["prediction_40_fits"] = time.perf_counter() - stage
display(prediction_metrics)
display(prediction["ensemble"])

In [ ]:
mouse_prediction = prediction["ensemble"].groupby("mouse", as_index=False).agg(
    true_infected=("true_infected", "first"), P_infected=("P_infected", "mean")
)
plt.scatter(mouse_prediction.mouse, mouse_prediction.P_infected,
            c=mouse_prediction.true_infected, cmap="coolwarm", vmin=0, vmax=1)
plt.axhline(0.5, color="grey", linestyle="--")
plt.ylabel("Held-out infection probability")
plt.xticks(rotation=45, ha="right")
plt.ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

## 6. Composition-only reference

Refit logistic regression using the retained C for each mouse fold.
Only training fractions enter its scaler and model. The inner C search
is omitted; the fold-specific selections are visible in the companion JSON.

In [ ]:
stage = time.perf_counter()
lr_prediction = lh.composition_lr(graphs, meta, SELECTED["composition_lr"])
lr_prediction.to_csv(partition_dir / "composition_lr_predictions.csv", index=False)
lr_metrics = lh.metric_table(lr_prediction, "Composition LR · selected fold C")
metrics = pd.concat([prediction_metrics, lr_metrics], ignore_index=True)
metrics.to_csv(partition_dir / "prediction_metrics.csv", index=False)
display(metrics)
TIMINGS["composition_lr"] = time.perf_counter() - stage

## 7. Explain every held-out ROI

Refit 40 separate models with the original patch-pooled objective for
interpretation. For all 24 ROIs, compute signed integrated gradients
towards the mean infected logit, using a zero baseline in standardized
feature space and 50 integration steps. Average signed attributions over
the five seeds, then sum their absolute values per node.

In [ ]:
# Explanation fits use the separate patch-pooled loss and public signed IG.
stage = time.perf_counter()
explanation = lh.run_outer_folds(
    graphs, meta, seeds=SEEDS, config=SELECTED["fixed_gnn"],
    objective="patch_sum", output_dir=partition_dir / "explanation",
    explain=True, steps=SELECTED["ig_steps"]
)
lh.save_outer_outputs(explanation, partition_dir / "explanation")
assert explanation["attributions"].roi.nunique() == 24
assert len(explanation["ig_residuals"]) == 24 * len(SEEDS)
TIMINGS["explanation_40_fits_120_ig"] = time.perf_counter() - stage
lh.show_ig(explanation["attributions"], roi, remodelled)

In [ ]:
localisation = descriptive.ig_localisation(
    explanation["attributions"], infection_cluster_id=remodelled
)
localisation.to_csv(partition_dir / "ig_localisation.csv", index=False)
display(localisation.query("is_infection_cluster"))
display(explanation["ig_residuals"].groupby("mouse").completeness_residual.agg(
    ["min", "max", "mean"]
))

## 8. Export timings and compare labelled references

The table below contains deposited paper values for context. The tables
above contain the new fits. Supplementary selections, controls and the
exact permutation test are available in the linked notebooks.

In [ ]:
reference = pd.DataFrame([
    {"analysis": "Fixed GraphSAGE, mouse accuracy", "paper_reference": SELECTED["paper_reference"]["fixed_mouse_accuracy"]},
    {"analysis": "Fixed GraphSAGE, mouse ROC-AUC", "paper_reference": SELECTED["paper_reference"]["fixed_mouse_auc"]},
    {"analysis": "Fixed GraphSAGE, correctly classified ROIs", "paper_reference": SELECTED["paper_reference"]["fixed_correct_rois"]},
    {"analysis": "Composition LR, mouse ROC-AUC", "paper_reference": SELECTED["paper_reference"]["composition_lr_mouse_auc"]},
])
display(reference)
TIMINGS["notebook_total"] = time.perf_counter() - started
timing_table = pd.DataFrame({"stage": list(TIMINGS), "seconds": list(TIMINGS.values())})
timing_table.to_csv(partition_dir / "timings.csv", index=False)
lh.write_json(partition_dir / "run.json", {
    "partition_mode": PARTITION_MODE, "config": SELECTED,
    "library_versions": lh.environment(), "device": DEVICE,
    "seeds": SEEDS, "n_rois": len(meta), "n_mice": meta.mouse.nunique(),
    "timings_seconds": TIMINGS,
})
display(timing_table)
print(f"Results: {partition_dir.resolve()}")